# InfoNCE Contrastive Loss for Multi-Specialist Grokking

Test the new contrastive consistency loss we added. This tests:

- L2 squared distance between same-input embeddings from different specialists (positives)
- Different inputs from different specialists (negatives) to prevent collapse

Cells:

| # | cell | seed | loss | temp | lambda | warmup | steps | purpose |
|---|---|---|---|---|---|---|---|---|
| 1 | `infonce_seed42` | 42 | infonce | 0.1 | 0.1 | 5000 | 25k | anchor |
| 2 | `infonce_seed43` | 43 | infonce | 0.1 | 0.1 | 5000 | 25k | seed robustness |
| 3 | `infonce_seed44` | 44 | infonce | 0.1 | 0.1 | 5000 | 25k | seed robustness |
| 4 | `infonce_seed45` | 45 | infonce | 0.1 | 0.1 | 5000 | 25k | seed robustness |
| 5 | `infonce_long` | 42 | infonce | 0.1 | 0.1 | 5000 | 100k | budget probe |
| 6 | `mse_seed42` | 42 | mse_logits | - | 0.1 | 5000 | 25k | baseline comparison |
| 7 | `kl_seed42` | 42 | kl_softmax | - | 0.1 | 5000 | 25k | baseline comparison |

Shared: M=4 specialists, disjoint shards, `consistency_domain=train_inputs_only`. All cells share: train_data_pct=50, max_lr=5e-4, weight_decay=0.1.

**Before running:** ensure the runtime is set to GPU (T4 or P100) and that internet is enabled.

## 1. Clone the repo

Uses latest commit with our InfoNCE changes.

In [ ]:
import os, subprocess, sys

# Use latest commit (our changes are already there)
GROK_REPO = os.environ.get('GROK_REPO', 'https://github.com/yazankb/grok.git')
WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.expanduser('~/grok_work')
os.makedirs(WORK, exist_ok=True)
REPO_DIR = os.path.join(WORK, 'grok')

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', GROK_REPO, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all', '--tags'], check=True)
# Use latest commit (don't pin - use whatever we have locally)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', 'main'], check=True)
print('repo at:', REPO_DIR)
print('commit :', subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD']).decode().strip())

## 2. Install dependencies

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mod', 'sympy'], check=True)
import torch, pytorch_lightning as pl
print('torch =', torch.__version__, '| lightning =', pl.__version__, '| cuda =', torch.cuda.is_available())

## 3. Run the InfoNCE sweep

In [ ]:
import json

STEPS = int(os.environ.get('STEPS', 25000))
SWEEP_NAME = os.environ.get('SWEEP_NAME', 'infonce_pilot_v1')
LOGDIR = os.environ.get('GROK_LOGDIR', os.path.join(WORK, 'consistency_runs'))
GPU = '0' if torch.cuda.is_available() else '-1'

env = os.environ.copy()
env['PYTHONPATH'] = REPO_DIR + ':' + env.get('PYTHONPATH', '')

CELLS_JSON = json.dumps([
    {
        'name': 'infonce_seed42', 'description': 'InfoNCE seed 42 anchor',
        'random_seed': 42, 'consistency_loss': 'infonce', 'consistency_lambda': 0.1,
        'consistency_warmup_steps': 5000, 'consistency_domain': 'train_inputs_only',
        'infonce_temperature': 0.1
    },
    {
        'name': 'infonce_seed43', 'description': 'InfoNCE seed 43 robustness',
        'random_seed': 43, 'consistency_loss': 'infonce', 'consistency_lambda': 0.1,
        'consistency_warmup_steps': 5000, 'consistency_domain': 'train_inputs_only',
        'infonce_temperature': 0.1
    },
    {
        'name': 'infonce_seed44', 'description': 'InfoNCE seed 44 robustness',
        'random_seed': 44, 'consistency_loss': 'infonce', 'consistency_lambda': 0.1,
        'consistency_warmup_steps': 5000, 'consistency_domain': 'train_inputs_only',
        'infonce_temperature': 0.1
    },
    {
        'name': 'infonce_seed45', 'description': 'InfoNCE seed 45 robustness',
        'random_seed': 45, 'consistency_loss': 'infonce', 'consistency_lambda': 0.1,
        'consistency_warmup_steps': 5000, 'consistency_domain': 'train_inputs_only',
        'infonce_temperature': 0.1
    },
    {
        'name': 'infonce_long_100k', 'description': 'InfoNCE 100k budget probe',
        'random_seed': 42, 'consistency_loss': 'infonce', 'consistency_lambda': 0.1,
        'consistency_warmup_steps': 5000, 'consistency_domain': 'train_inputs_only',
        'infonce_temperature': 0.1, 'consistency_steps': 100000
    },
    {
        'name': 'infonce_temp01', 'description': 'InfoNCE temp 0.01 ablation',
        'random_seed': 42, 'consistency_loss': 'infonce', 'consistency_lambda': 0.1,
        'consistency_warmup_steps': 5000, 'consistency_domain': 'train_inputs_only',
        'infonce_temperature': 0.01
    },
    {
        'name': 'infonce_temp1', 'description': 'InfoNCE temp 1.0 ablation',
        'random_seed': 42, 'consistency_loss': 'infonce', 'consistency_lambda': 0.1,
        'consistency_warmup_steps': 5000, 'consistency_domain': 'train_inputs_only',
        'infonce_temperature': 1.0
    },
])

cells_json_path = os.path.join(WORK, 'sweep_cells_infonce.json')
with open(cells_json_path, 'w') as f:
    f.write(CELLS_JSON)

cmd = [
    sys.executable,
    os.path.join(REPO_DIR, 'scripts', 'run_consistency_sweep.py'),
    '--sweep_name', SWEEP_NAME,
    '--logdir', LOGDIR,
    '--consistency_steps', str(STEPS),
    '--gpu', GPU,
    '--n_models', '4',
    '--sharding', 'disjoint',
    '--train_data_pct', '50',
    '--max_lr', '5e-4',
    '--weight_decay', '0.1',
    '--shard_batch_size', '256',
    '--consistency_batch_size', '256',
    '--eval_every', '500',
    '--checkpoint_every', '5000',
    '--log_every', '50',
    '--cells_json', cells_json_path,
]
print('running:', ' '.join(cmd))
subprocess.run(cmd, check=True, env=env, cwd=REPO_DIR)

## 4. Inspect results

In [ ]:
import json

summary_path = os.path.join(LOGDIR, SWEEP_NAME, 'sweep_summary.json')
with open(summary_path) as f:
    summary = json.load(f)

print(f"sweep: {summary['sweep_name']}  ({len(summary['cells'])} cell(s))")
print(f"started: {summary.get('started_at')}  finished: {summary.get('finished_at')}")
print()
fmt = '{:<22} {:<8} {:>10} {:>12} {:>10}'
print(fmt.format('cell', 'status', 'best_ens', 'best_merged', 'sec'))
print('-' * 64)
for c in summary['cells']:
    r = c.get('results', {}) or {}
    print(fmt.format(
        c['name'], c['status'],
        f"{r.get('best_val_acc_ensemble', float('nan')):.2f}" if 'best_val_acc_ensemble' in r else 'n/a',
        f"{r.get('best_val_acc_merged', float('nan')):.2f}" if 'best_val_acc_merged' in r else 'n/a',
        f"{c.get('elapsed_sec', 0):.0f}",
    ))

## 5. Plot per-cell training curves

In [ ]:
import csv
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
(ax_ens, ax_merged), (ax_kl, ax_ent) = axes

for c in summary['cells']:
    if c['status'] != 'ok':
        continue
    eval_csv = os.path.join(c['experiment_dir'], 'consistency_eval.csv')
    if not os.path.isfile(eval_csv):
        continue
    with open(eval_csv) as f:
        rows = list(csv.DictReader(f))
    steps = [int(r['step']) for r in rows]
    ax_ens.plot(steps, [float(r['val_acc_ensemble']) for r in rows], label=c['name'])
    ax_merged.plot(steps, [float(r['val_acc_merged']) for r in rows], label=c['name'])
    ax_kl.plot(steps, [float(r['pairwise_kl_val_mean']) for r in rows], label=c['name'])
    ax_ent.plot(steps, [float(r['unsup_entropy_mean']) for r in rows], label=c['name'])

for ax, title in [
    (ax_ens, 'Ensemble val acc (%)'),
    (ax_merged, 'Merged val acc (%)'),
    (ax_kl, 'Pairwise val KL (alignment)'),
    (ax_ent, 'Unsup output entropy (collapse diag)'),
]:
    ax.set_title(title)
    ax.set_xlabel('step')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
ax_kl.set_yscale('log')
fig.tight_layout()
out_png = os.path.join(LOGDIR, SWEEP_NAME, 'sweep_curves.png')
fig.savefig(out_png, dpi=120)
print('saved:', out_png)
plt.show()

## 6. Bundle artifacts for download

In [ ]:
import shutil
sweep_dir = os.path.join(LOGDIR, SWEEP_NAME)
out_zip = os.path.join(WORK, f'{SWEEP_NAME}.zip')
if os.path.isfile(out_zip):
    os.remove(out_zip)
shutil.make_archive(out_zip[:-4], 'zip', root_dir=sweep_dir)
print('bundle:', out_zip, '|', os.path.getsize(out_zip) / 1e6, 'MB')